# anthropic — three input rates on one call

Anthropic's prompt caching bills uncached input, cache **reads** and cache **writes** at three different rates. Getting the cost right by hand is error-prone; a two-rate approximation is simply wrong.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## The five steps

Every recipe in `providers/` walks the same five, in the same order:

| # | Step | Here |
|---|---|---|
| 1 | **connect** | `Anthropic()` — the `messages.create` shape |
| 2 | **instrument** | one wrap — detection is structural, not name-based |
| 3 | **govern** | a `tokenguard` budget **and** a `guardrails` gate |
| 4 | **record** | `cassette` — the same call replayed offline, 0 provider calls |
| 5 | **prove** | `acttrace` `verify()` and a cost that came from `prices` |

⚠️ **Pre-flight token counting for Claude is approximate** — `o200k` under-counts Claude by a measured **1.49×** (English) / **1.14×** (code). The usage below is *settled* usage, which is exact.

## 1–3 · Connect, instrument, call

In [ ]:
import main as recipe
from cendor.core import bus, instrument
from cendor.core.types import LLMCall

seen, calls = [], []
bus.subscribe(lambda e: calls.append(e) if isinstance(e, LLMCall) else None)
client = instrument(recipe.fake_anthropic(seen))
client.messages.create(
    model=recipe.MODEL,
    max_tokens=256,
    messages=[{"role": "user", "content": "Answer using the cached system prompt."}],
)
u = calls[-1].usage
print(
    f"usage: {u.input_tokens:,} in ({u.cached_tokens} cache-read)"
    f" + {u.cache_write} cache-write -> {u.output_tokens} out"
)
print(f"cost : ${calls[-1].cost.amount}")

## 5 · Prove it

In [ ]:
assert u.cached_tokens == 800, "cache READS were not normalized into the input subset"
assert u.cache_write == 300, "cache WRITES were not tracked as their own billed category"
assert calls[-1].cost.amount > 0
print("OK — uncached + cache-read + cache-write + output")